# पाठ 13 - एजेंट मेमोरी


## Setup

यह नोटबुक दिखाता है कि कैसे **Microsoft Agent Framework** (MAF) का उपयोग करके **स्थायी मेमोरी** के साथ एक ट्रैवल बुकिंग एजेंट बनाया जाए।

आप सीखेंगे कि एजेंट की मेमोरी के विभिन्न प्रकार — वर्किंग, शॉर्ट-टर्म, और लॉंग-टर्म — बातचीत के दौरान जानकारी को कैसे संजोते और उपयोग करते हैं।

**पूर्वापेक्षाएँ:**
- एक Azure AI Foundry प्रोजेक्ट जिसमें एक तैनात चैट मॉडल हो (जैसे `gpt-4o-mini`)।
- Azure CLI में लॉग इन किया हुआ — अपने टर्मिनल में `az login` चलाएँ।
- `AZURE_AI_PROJECT_ENDPOINT` — आपका Azure AI Foundry प्रोजेक्ट एंडपॉइंट।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपके तैनात मॉडल का नाम।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजेंट मेमोरी के प्रकार

एआई एजेंट विभिन्न प्रकार की मेमोरी का लाभ उठा सकते हैं, जो प्रत्येक एक अलग उद्देश्य पूरा करती है:

### वर्किंग मेमोरी
स्वयं वार्तालाप थ्रेड — एक ही सेशन में आदान-प्रदान किए गए संदेश। एजेंट एक ही थ्रेड में पहले के संदेशों का संदर्भ लेकर सहमति बनाए रख सकता है। MAF में इसे **`agent.create_session()`** के माध्यम से बनाया जाता है, जो एक `AgentSession` लौटाता है।

### शॉर्ट-टर्म मेमोरी
सूचना जो एक कार्य या सेशन की अवधि तक बनी रहती है लेकिन स्थायी रूप से संग्रहित नहीं होती। उदाहरण के लिए, एजेंट एक मल्टी-टर्न योजना वार्तालाप के दौरान तथ्य एकत्र कर सकता है और उन्हें अंतिम यात्रा कार्यक्रम बनाने के लिए उपयोग कर सकता है।

### लॉन्ग-टर्म मेमोरी
पसंद और तथ्य जो **सेशनों के बीच** बने रहते हैं। लौटने वाले उपयोगकर्ता को अपनी आहार संबंधी सीमाएं या यात्रा शैली दोहरानी नहीं पड़नी चाहिए। लॉन्ग-टर्म मेमोरी आमतौर पर बाहरी स्टोर — एक डेटाबेस, फाइल, या वेक्टर इंडेक्स — द्वारा समर्थित होती है और टूल्स के माध्यम से एजेंट को प्रदान की जाती है।


## सेशन्स के साथ वर्किंग मेमोरी

सबसे सरल मेमोरी का रूप है बातचीत सत्र। जब आप एक ही सत्र (जो `agent.create_session()` के माध्यम से बनाया गया हो) को लगातार `agent.run()` कॉल्स में पास करते हैं, तो एजेंट उस बातचीत का पूरा इतिहास देख सकता है और पहले के विवरणों को याद रख सकता है।

आइए एक ट्रैवल एजेंट बनाएं और वर्किंग मेमोरी का प्रदर्शन करें।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजेंट ने बजट को सही ढंग से याद किया क्योंकि दोनों संदेशों का वही सत्र साझा था। इसे **कार्यशील स्मृति** कहा जाता है — यह केवल सत्र के जीवनकाल के लिए मौजूद रहती है।

### नई थ्रेड के साथ क्या होता है?

यदि हम एक **नया** सत्र बनाते हैं, तो एजेंट को पिछली बातचीत की कोई याददाश्त नहीं होती:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालिक स्मृति पैटर्न

उपयोगकर्ता की प्राथमिकताओं को **सत्रों के दौरान** याद रखने के लिए, हमें एक स्थायी संग्रह की आवश्यकता होती है जो वार्तालाप थ्रेड के बाहर रहता है। एजेंट इस संग्रह तक पहुँचने के लिए **टूल्स** का उपयोग करता है — वे फ़ंक्शन जिन्हें वह जानकारी सहेजने और पुनः प्राप्त करने के लिए कॉल कर सकता है।

नीचे हमने एक सरल इन-मेमोरी प्राथमिकता संग्रह लागू किया है (उत्पादन में आप इसे एक डेटाबेस या वेक्टर इंडेक्स के साथ बैक कर सकते हैं) और इसे ऐसे टूल्स के रूप में प्रस्तुत किया है जिन्हें एजेंट उपयोग कर सकता है।

### संरचना
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — पहली बार उपयोगकर्ता सालगिरह की यात्रा बुक करता है

सारा पहली बार आती है। एजेंट को उपकरणों के माध्यम से उसकी प्राथमिकताओं को संग्रहीत करना चाहिए और उनके आधार पर होटलों की सिफारिश करनी चाहिए।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिदृश्य 2 — सारा हफ्तों बाद वापस आती है

सारा एक **नई थ्रेड** शुरू करती है (नए सत्र का अनुकरण करते हुए)। कार्यशील मेमोरी खाली है, लेकिन दीर्घकालिक प्राथमिकता भंडार में अभी भी उसकी जानकारी है। एजेंट को इसे पुनः प्राप्त करना चाहिए और सिफारिशों को वैयक्तिकृत करने के लिए इसका उपयोग करना चाहिए।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

इस पाठ में आपने एजेंट मेमोरी के तीन प्रकार और Microsoft Agent Framework के साथ उन्हें कैसे लागू करें, यह सीखा:

| मेमोरी प्रकार | MAF मैकेनिज्म | जीवनकाल |
|---|---|---|
| **वर्किंग** | `agent.create_session()` | एकल वार्तालाप |
| **शॉर्ट-टर्म** | थ्रेड के भीतर संचयी संदर्भ | एकल कार्य / सत्र |
| **लॉन्ग-टर्म** | बाहरी स्टोर जिसे `@tool` फ़ंक्शन के माध्यम से एक्सेस किया जाता है | सत्रों के बीच |

### मुख्य बिंदु
1. **`agent.create_session()`** वर्किंग मेमोरी प्रदान करता है — एजेंट एक सत्र के भीतर पूरी वार्तालाप इतिहास देखता है।
2. **नए सत्र संदर्भ खो देते हैं** — बिना लॉन्ग-टर्म मेमोरी के एजेंट पिछली वार्तालापों को याद नहीं रख सकता।
3. **`@tool` फ़ंक्शन** गैप को पाटते हैं — वे एजेंट को जानकारी को एक स्थायी स्टोर में सहेजने और पुनः प्राप्त करने देते हैं।
4. **व्यक्तिगतकरण समय के साथ सुधारता है** — जितनी अधिक पसंदे सहेजी जाती हैं, एजेंट की सिफारिशें उतनी ही बेहतर होती हैं।

### वास्तविक दुनिया के आवेदन
- **कस्टमर सर्विस**: ग्राहक का इतिहास और पसंद याद रखें
- **पर्सनल असिस्टेंट्स**: दिनों या हफ्तों के बीच संदर्भ बनाए रखें
- **हेल्थकेयर**: रोगी की जानकारी और पसंद ट्रैक करें
- **ई-कॉमर्स**: इतिहास के आधार पर व्यक्तिगत शॉपिंग

### अगले कदम
- इन-मेमोरी डिक्ट को डेटाबेस या वेक्टर स्टोर (जैसे Azure AI Search) से बदलें
- समय-संवेदनशील जानकारी के लिए मेमोरी समाप्ति जोड़ें
- साझा मेमोरी वाले मल्टी-एजेंट सिस्टम बनाएँ
- नॉलेज-ग्राफ-बैक्ड मेमोरी के लिए Cognee नोटबुक का अन्वेषण करें


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यह दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। यद्यपि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में गलतियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ को इसकी मूल भाषा में प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या मिसइंटरप्रिटेशन के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
